# CT-378 Machine Learning: Low-Water Crop Selection Under Drought
**Group Members:** Muhammad Afzal (CT-24263), Syed Muhammad Hunain (CT-24281), Ayesha (CT-24309)

---
## Task 1: Data Ingestion, Merging & Validation
Combine raw agricultural, meteorological, and irrigation datasets for Pakistan districts (2000–2023).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 30)
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
# Resolve project root regardless of working directory
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")

# Paths to raw parquet files
path_crops = PROJECT_ROOT / "data/raw/crop_production_and_acreage/district_crop_production_and_yields.parquet"
path_climate = PROJECT_ROOT / "data/raw/rainfall_and_drought_indices/district_rainfall_deficit_and_soil_moisture.parquet"
path_water = PROJECT_ROOT / "data/raw/water_resource_availability/irrigation_and_barani_water_regimes.parquet"

df_crops = pd.read_parquet(path_crops)
df_climate = pd.read_parquet(path_climate)
df_water = pd.read_parquet(path_water)

print(f"Crops table: {df_crops.shape}")
print(f"Climate table: {df_climate.shape}")
print(f"Water regimes table: {df_water.shape}")

Project root: /home/prx/Projects/machine-learning/climate-machine-learning
Crops table: (11088, 9)
Climate table: (2448, 15)
Water regimes table: (2448, 8)


In [3]:
# Merge datasets on key temporal and geographic columns
join_keys = ["District", "Province", "Crop_Year", "Season"]
df_merged = pd.merge(df_crops, df_climate, on=join_keys, how="inner")
df = pd.merge(df_merged, df_water, on=join_keys, how="inner")

print(f"Merged DataFrame Shape: {df.shape}")
assert df.shape == (11088, 24), f"Unexpected shape: {df.shape}"
print("Shape assertion passed: 11,088 rows, 24 columns.")

Merged DataFrame Shape: (11088, 24)
Shape assertion passed: 11,088 rows, 24 columns.


In [4]:
# Missing value check
missing_counts = df.isnull().sum()
print("Missing values per column:")
print(missing_counts[missing_counts > 0] if (missing_counts > 0).any() else "No missing values found across all columns.")

Missing values per column:
No missing values found across all columns.


In [5]:
# Validate candidate low-water (drought-resilient) crops vs high-water crops
crops_summary = df[["Crop_Name", "Is_Drought_Resilient", "Crop_Water_Demand_mm", "Season"]].drop_duplicates().sort_values("Crop_Water_Demand_mm")
print("Crops in dataset with water demand and drought resilience:")
print(crops_summary.to_string(index=False))

low_water_crops = crops_summary[crops_summary["Is_Drought_Resilient"] == True]["Crop_Name"].tolist()
print(f"\nIdentified {len(low_water_crops)} low-water candidate crops: {low_water_crops}")

Crops in dataset with water demand and drought resilience:
           Crop_Name  Is_Drought_Resilient  Crop_Water_Demand_mm Season
     Chickpea (Gram)                  True                 220.0   Rabi
Pearl Millet (Bajra)                  True                 250.0 Kharif
        Barley (Jau)                  True                 260.0   Rabi
    Canola / Mustard                  True                 280.0   Rabi
     Sorghum (Jowar)                  True                 300.0 Kharif
               Wheat                 False                 450.0   Rabi
               Maize                 False                 550.0 Kharif
              Cotton                 False                 700.0 Kharif
 Rice (Basmati/IRRI)                 False                1200.0 Kharif
           Sugarcane                 False                1800.0 Kharif

Identified 5 low-water candidate crops: ['Chickpea (Gram)', 'Pearl Millet (Bajra)', 'Barley (Jau)', 'Canola / Mustard', 'Sorghum (Jowar)']
